<a href="https://colab.research.google.com/github/Jaguar838/data-engineering-zoomcamp/blob/main/HW/2026/hw07/hw-07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 7 Homework

In [ ]:
!python --version

Python 3.12.3


In this homework, we'll practice streaming with Kafka (Redpanda) and PyFlink.

We use Redpanda, a drop-in replacement for Kafka. It implements the same
protocol, so any Kafka client library works with it unchanged.

For this homework we will be using Green Taxi Trip data from October 2025:

- [green_tripdata_2025-10.parquet](https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet)


## Setup

We'll use the same infrastructure from the [workshop](../../../07-streaming/workshop/).

Follow the setup instructions: build the Docker image, start the services:

```bash
cd 07-streaming/workshop/
docker compose build
docker compose up -d
```

This gives us:

- Redpanda (Kafka-compatible broker) on `localhost:9092`
- Flink Job Manager at http://localhost:8081
- Flink Task Manager
- PostgreSQL on `localhost:5432` (user: `postgres`, password: `postgres`)

If you previously ran the workshop and have old containers/volumes,
do a clean start:

```bash
docker compose down -v
docker compose build
docker compose up -d
```

Note: the container names (like `workshop-redpanda-1`) assume the
directory is called `workshop`. If you renamed it, adjust accordingly.

In [ ]:
!uvx pgcli -h localhost -p 5432 -U postgres -d postgres

Installed 14 packages in 34ms2                                       
connection is bad: connection to server at "::1", port 5432 failed: Cannot assign requested address
	Is the server running on that host and accepting TCP/IP connections?
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: '5432', hostaddr: '127.0.0.1': connection failed: connection to server at "127.0.0.1", port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
- host: 'localhost', port: '5432', hostaddr: '::1': connection is bad: connection to server at "::1", port 5432 failed: Cannot assign requested address
	Is the server running on that host and accepting TCP/IP connections?


## Q1: Redpanda version

Now let's find out the version of redpandas.

For that, check the output of the command `rpk help` _inside the container_. The name of the container is `redpanda`.

Find out what you need to execute based on the `help` output.

What's the version, based on the output of the command you executed? (copy the entire version)

In [ ]:
!docker compose exec redpanda rpk version

service "redpanda" is not running


Q1 answer: rpk version: v25.3.9

## Question 2. Sending data to Redpanda

Create a topic called `green-trips`:

```bash
docker exec -it redpanda rpk topic create green-trips
```

Now write a producer to send the green taxi data to this topic.

Read the parquet file and keep only these columns:

- `lpep_pickup_datetime`
- `lpep_dropoff_datetime`
- `PULocationID`
- `DOLocationID`
- `passenger_count`
- `trip_distance`
- `tip_amount`
- `total_amount`

Convert each row to a dictionary and send it to the `green-trips` topic.
You'll need to handle the datetime columns - convert them to strings
before serializing to JSON.

Measure the time it takes to send the entire dataset and flush:

```python
from time import time

t0 = time()

# send all rows ...

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')
```

How long did it take to send the data?

- 10 seconds
- 60 seconds
- 120 seconds
- 300 seconds

In [ ]:
!docker compose exec redpanda rpk  topic create green-trips

TOPIC        STATUS
green-trips  OK


In [ ]:
!docker compose exec redpanda rpk  topic list

NAME         PARTITIONS  REPLICAS
green-trips  1           1


In [ ]:
columns = [
    'lpep_pickup_datetime',
    'lpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'tip_amount',
    'total_amount'
]

In [ ]:
import pandas as pd
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet'
df = pd.read_parquet(url, columns=columns)
df.head(5)

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,passenger_count,trip_distance,tip_amount,total_amount
0,2025-10-01 00:21:47,2025-10-01 00:24:37,247,69,1.0,0.70,1.70,10.00
1,2025-10-01 00:14:03,2025-10-01 00:24:14,66,25,1.0,1.61,2.78,16.68
2,2025-10-01 00:16:44,2025-10-01 00:16:47,244,244,1.0,0.00,2.20,13.20
3,2025-10-01 00:07:36,2025-10-01 00:32:14,95,170,1.0,10.37,11.31,67.85
4,2025-09-30 21:10:29,2025-09-30 21:22:30,82,138,1.0,4.07,6.82,34.12


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49416 entries, 0 to 49415
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   lpep_pickup_datetime   49416 non-null  datetime64[us]
 1   lpep_dropoff_datetime  49416 non-null  datetime64[us]
 2   PULocationID           49416 non-null  int32         
 3   DOLocationID           49416 non-null  int32         
 4   passenger_count        44401 non-null  float64       
 5   trip_distance          49416 non-null  float64       
 6   tip_amount             49416 non-null  float64       
 7   total_amount           49416 non-null  float64       
dtypes: datetime64[us](2), float64(4), int32(2)
memory usage: 2.6 MB


In [ ]:
from kafka import KafkaProducer
from models_green import GreenTrip, trip_from_row, to_json

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=to_json
)

In [ ]:
from time import time
from tqdm.auto import tqdm

In [ ]:
TOPIC_NAME = 'green-trips'
print(f"Sending {len(df)} records to Kafka topic '{TOPIC_NAME}'...")

t0 = time()

# Send all rows
for row_data in tqdm(df.to_dict(orient='records'), desc="Sending to Kafka"):
    # Create a GreenTrip object from the row
    trip = trip_from_row(row_data)
    # The value_serializer in the producer will automatically call to_json(trip)
    producer.send(TOPIC_NAME, value=trip)

# Ensure all messages are sent
producer.flush()

t1 = time()

print(f'took {(t1 - t0):.2f} seconds')

# Gracefully close the producer
producer.close()

print("Finished sending data.")

Sending 49416 records to Kafka topic 'green-trips'...


Sending to Kafka:   0%|          | 0/49416 [00:00<?, ?it/s]

took 10.35 seconds
Finished sending data.


In [ ]:
!docker compose exec redpanda rpk topic consume green-trips -p 0 -o -1 -n 1

{
  "topic": "green-trips",
  "value": "{\"lpep_pickup_datetime\": \"2025-10-31 23:23:00\", \"lpep_dropoff_datetime\": \"2025-10-31 23:37:00\", \"PULocationID\": 195, \"DOLocationID\": 33, \"passenger_count\": NaN, \"trip_distance\": 3.0, \"tip_amount\": 0.0, \"total_amount\": 19.6}",
  "timestamp": 1773302216027,
  "partition": 0,
  "offset": 49415
}


Q2 answer: 10 seconds

## Question 3. Consumer - trip distance

Write a Kafka consumer that reads all messages from the `green-trips` topic
(set `auto_offset_reset='earliest'`).

Count how many trips have a `trip_distance` greater than 5.0 kilometers.

How many trips have `trip_distance` > 5?

- 6506
- 7506
- 8506
- 9506

In [ ]:
import json
from kafka import KafkaConsumer

# --- Consumer Logic ---
print("Initializing Kafka consumer...")

# The consumer will stop iterating if no new messages arrive for `consumer_timeout_ms` milliseconds.
consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',  # Start from the beginning of the topic
    consumer_timeout_ms=5000,      # Stop if no messages for 5 seconds
    value_deserializer=lambda v: json.loads(v.decode('utf-8')) # Decode JSON
)

print(f"Reading messages from topic '{TOPIC_NAME}' to count long trips...")

count_long_trips = 0

try:
    # Using tqdm to show progress, especially if there are many messages
    for message in tqdm(consumer, desc="Processing messages"):
        # message.value is a dictionary thanks to the deserializer
        trip_distance = message.value.get('trip_distance', 0)

        if trip_distance > 5.0:
            count_long_trips += 1

    print("Finished reading messages from the topic.")

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # This will be the answer to the question
    print(f"How many trips have trip_distance > 5? Answer: {count_long_trips}")

    consumer.close()
    print("Consumer closed.")

Initializing Kafka consumer...
Reading messages from topic 'green-trips' to count long trips...


Processing messages: 0it [00:00, ?it/s]

Finished reading messages from the topic.
How many trips have trip_distance > 5? Answer: 8506
Consumer closed.


Q3 answer: 8506

## Part 2: PyFlink (Questions 4-6)

For the PyFlink questions, you'll adapt the workshop code to work with
the green taxi data. The key differences from the workshop:

- Topic name: `green-trips` (instead of `rides`)
- Datetime columns use `lpep_` prefix (instead of `tpep_`)
- You'll need to handle timestamps as strings (not epoch milliseconds)

You can convert string timestamps to Flink timestamps in your source DDL:

```sql
lpep_pickup_datetime VARCHAR,
event_timestamp AS TO_TIMESTAMP(lpep_pickup_datetime, 'yyyy-MM-dd HH:mm:ss'),
WATERMARK FOR event_timestamp AS event_timestamp - INTERVAL '5' SECOND
```

Before running the Flink jobs, create the necessary PostgreSQL tables
for your results.

Important notes for the Flink jobs:

- Place your job files in `workshop/src/job/` - this directory is
  mounted into the Flink containers at `/opt/src/job/`
- Submit jobs with:
  `docker exec -it workshop-jobmanager-1 flink run -py /opt/src/job/your_job.py`
- The `green-trips` topic has 1 partition, so set parallelism to 1
  in your Flink jobs (`env.set_parallelism(1)`). With higher parallelism,
  idle consumer subtasks prevent the watermark from advancing.
- Flink streaming jobs run continuously. Let the job run for a minute
  or two until results appear in PostgreSQL, then query the results.
  You can cancel the job from the Flink UI at http://localhost:8081
- If you sent data to the topic multiple times, delete and recreate
  the topic to avoid duplicates:
  `docker exec -it workshop-redpanda-1 rpk topic delete green-trips`

## Question 4. Tumbling window - pickup location

Create a Flink job that reads from `green-trips` and uses a 5-minute
tumbling window to count trips per `PULocationID`.

Write the results to a PostgreSQL table with columns:
`window_start`, `PULocationID`, `num_trips`.

After the job processes all data, query the results:

```sql
SELECT PULocationID, num_trips
FROM <your_table>
ORDER BY num_trips DESC
LIMIT 3;
```

Which `PULocationID` had the most trips in a single 5-minute window?

- 42
- 74
- 75
- 166

In [ ]:
!docker compose exec jobmanager ./bin/flink run -py /opt/src/job/tumbling_window_job.py --pyFiles /opt/src

Starting PyFlink Tumbling Window job...
Job has been submitted with JobID 5a3ac03b4f8afc4f101b60bdd6b4c17b


Q4 answer: 74

## Question 5. Session window - longest streak

Create another Flink job that uses a session window with a 5-minute gap
on `PULocationID`, using `lpep_pickup_datetime` as the event time
with a 5-second watermark tolerance.

A session window groups events that arrive within 5 minutes of each other.
When there's a gap of more than 5 minutes, the window closes.

Write the results to a PostgreSQL table and find the `PULocationID`
with the longest session (most trips in a single session).

How many trips were in the longest session?

- 12
- 31
- 51
- 81

In [ ]:
!docker compose exec jobmanager ./bin/flink run -py /opt/src/job/session_job.py --pyFiles /opt/src

^C


Q5 answer: 81


## Question 6. Tumbling window - largest tip

Create a Flink job that uses a 1-hour tumbling window to compute the
total `tip_amount` per hour (across all locations).

Which hour had the highest total tip amount?

- 2025-10-01 18:00:00
- 2025-10-16 18:00:00
- 2025-10-22 08:00:00
- 2025-10-30 16:00:00


In [ ]:
!docker compose exec jobmanager ./bin/flink run -py /opt/src/job/sliding_window_job.py --pyFiles /opt/src

Starting PyFlink Sliding Window job...
Job has been submitted with JobID 39fd43a0372d93b41f6a6aa7e200e7dd
^C

------------------------------------------------------------
 The program finished with the following exception:

org.apache.flink.client.program.ProgramInvocationException: The main method caused an error: Shutdown in progress
	at org.apache.flink.client.program.PackagedProgram.callMainMethod(PackagedProgram.java:360)
	at org.apache.flink.client.program.PackagedProgram.invokeInteractiveModeForExecution(PackagedProgram.java:223)
	at org.apache.flink.client.ClientUtils.executeProgram(ClientUtils.java:105)
	at org.apache.flink.client.cli.CliFrontend.executeProgram(CliFrontend.java:1017)
	at org.apache.flink.client.cli.CliFrontend.run(CliFrontend.java:230)
	at org.apache.flink.client.cli.CliFrontend.parseAndRun(CliFrontend.java:1261)


Q5 answer: 2025-10-16 18:00:00